# hyperloom tour

Feature demonstrations only. For correctness, layout-quality diagnostics, bounded stress tests, and task-based evaluation use **evaluation.ipynb**. Run viewers one at a time; request a handle and call `handle.close()` to release existing sessions.

This tour keeps only the most recent viewer live and closes the previous session automatically.


## Setup

This repo isn't published yet, so point at the local packages directly. If you've installed `hyperloom-core`/`hyperloom-bridge` into your kernel's environment (see the repo README), you can skip the `sys.path` lines.

**Environment.** Run this notebook with the repo's Python environment, which has every dependency (NumPy, msgpack, websockets, the optional pandas/networkx loaders, and the notebook and benchmark tools): `uv sync`, or `python -m venv .venv && .venv/bin/pip install -r requirements-dev.txt`, then pick the `.venv` interpreter as the notebook kernel. The `sys.path` lines below only make the local packages importable; they do not install dependencies, so a kernel from a different environment fails at `import numpy`.

In [5]:
import sys
from pathlib import Path

repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "packages/core/hyperloom_core").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Start Jupyter inside the hyperloom checkout")
for path in (repo_root, repo_root / "packages/core", repo_root / "packages/bridge"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from hyperloom_bridge import show as _show
from hyperloom_core import Graph
import numpy as np

# Keep one live viewer at a time when rerunning the tour.
if "_active_view" in globals() and _active_view is not None:
    _active_view.close()
_active_view = None

def show(graph, **kwargs):
    global _active_view
    if _active_view is not None:
        _active_view.close()
    _active_view = _show(graph, return_handle=True, **kwargs)


## Plain graph

Nodes + edges, force-directed layout streaming in as it converges.

In [ ]:
g = Graph()
for i in range(20):
    g.add_node(i)
for i in range(19):
    g.add_edge(i, i + 1)
for i in range(0, 19, 4):
    g.add_edge(i, (i + 9) % 20)

show(g, seed=0)

## Directed graph

Arrowheads point from source to target.

In [ ]:
g = Graph()
for i in range(12):
    g.add_node(i)
for i in range(12):
    g.add_edge(i, (i + 1) % 12, directed=True)
for i in range(0, 12, 3):
    g.add_edge(i, (i + 5) % 12, directed=True)

show(g, seed=1)

## Temporal graph

Temporal networks usually arrive as **contact sequences**: one row per event, `u v t`. Load them with `from_temporal_edgelist` (or `read_temporal_edgelist("events.txt")` for a file, `from_pandas_temporal_edgelist` for a DataFrame); no `t_start`/`t_end` needed. Interval data works too (`intervals=True` with `(u, v, start, end)` rows).

The timeline slider at the bottom (click **All time** to start scrubbing) has a selector: **Trailing window** (the default for contact sequences) shows events from the last N time units, **At this moment** shows only what is active at exactly `t`, and **Everything so far** accumulates. For a view of the *whole* timeline at once, click **Time ribbon** (top-left): each time-bucket draws as its own separated plane (blue = early, orange = late), with aligned node positions across buckets.


In [ ]:
from hyperloom_core import from_temporal_edgelist

# A contact sequence: (u, v, t) rows, exactly as temporal datasets are published.
events = [(i % 12, (i * 5 + 3) % 12, i * 2) for i in range(48)]
events += [(i, (i + 1) % 12, 0) for i in range(12)]   # a ring of early contacts
g = from_temporal_edgelist(events)

show(g, seed=3)


### Temporal data from a file or DataFrame

Real event data usually lives in a text file or a table, with dates rather than small integers. `read_temporal_edgelist` reads whitespace- or delimiter-separated `u v t` lines (ISO dates are understood), and `from_pandas_temporal_edgelist` reads a DataFrame. Graphs loaded from dates show **dates** on the timeline and ribbon, and the trailing window is set in days.

Options worth knowing: `duration=` gives each contact a lifetime, `intervals=True` reads `(u, v, start, end)` rows, `attrs={"name": column}` keeps extra columns on each event, and `time_format="%d/%m/%Y"` reads other date layouts. **Unix timestamps** (seconds since 1970, very common) are plain numbers, so say so with `time_unit="epoch_seconds"` (or `"epoch_milliseconds"`, ...): they are converted and shown as dates. Every loader takes `time_unit` and `time_format`; numbers that look like Unix time without `time_unit` trigger a warning.

In the **Time ribbon**, the bar at the bottom sets the number of buckets and the split: *Equal time* (same duration per panel) or *Equal number of events* (same activity per panel, so busy periods get narrow panels). The legend shows each panel's event count. Do not use `from_edgelist` for `(u, v, t)` rows: its third value is a weight.

In [ ]:
import tempfile
from datetime import datetime, timedelta
from hyperloom_core import read_temporal_edgelist

# A small event log in the usual "u v timestamp" text format (one line per interaction).
start = datetime(2024, 3, 1, 9, 0)
lines = ["# who  with  when"]
rng = np.random.default_rng(4)
people = ["ana", "ben", "chi", "dev", "eli", "fay", "gus", "hana", "ivo", "jo"]
for hour in range(0, 24 * 14, 3):                       # two weeks, an event every three hours
    a, b = rng.choice(people, size=2, replace=False)
    lines.append(f"{a} {b} {(start + timedelta(hours=hour)).strftime('%Y-%m-%d %H:%M:%S')}")
events_file = Path(tempfile.mkdtemp()) / "events.txt"
events_file.write_text("\n".join(lines))

g = read_temporal_edgelist(events_file)
print(g.num_nodes, "people,", len(list(g.connectors())), "events, time_unit =", g.time_unit)
show(g, seed=2)   # click "All time", then drag the slider: the label shows dates and the window is in days

In [ ]:
# The same data from a DataFrame (requires pandas), keeping a column as an event attribute.
try:
    import pandas as pd
    from hyperloom_core import from_pandas_temporal_edgelist
    df = pd.DataFrame({
        "src": ["ana", "ben", "chi", "ana"],
        "dst": ["ben", "chi", "dev", "dev"],
        "when": pd.to_datetime(["2024-03-01 09:00", "2024-03-02 12:00", "2024-03-05 08:30", "2024-03-09 17:45"]),
        "channel": ["mail", "chat", "chat", "mail"],
    })
    g = from_pandas_temporal_edgelist(df, "src", "dst", "when", edge_attr=["channel"], duration=2 * 86400)
    print([(c.attrs["channel"], c.t_start, c.t_end) for c in g.connectors()][:2])
except ImportError:
    print("pandas is not installed; skipping the DataFrame example")

# Unix timestamps are plain numbers: declare the unit and the viewer shows dates.
unix = [("ana", "ben", 1709283600), ("ben", "chi", 1709456400), ("chi", "dev", 1709629200)]
print(from_temporal_edgelist(unix, time_unit="epoch_seconds").time_unit)


### A real event stream: Reddit hyperlinks

If you have the SNAP *Reddit Hyperlink Network* (`soc-redditHyperlinks-title.tsv`, 368 MB) in `sample_data/`, this loads all 572K timestamped hyperlinks between 54K subreddits (about two seconds) and opens them in the viewer. Text timestamps, repeated pairs and a subreddit named with a 20-digit number are all handled. Use the timeline's trailing window to watch activity move through 2014-2017. Sentiment is kept as an event attribute. See `evaluation.ipynb` for the measured checks.

In [ ]:
RUN_REDDIT_DEMO = False   # needs sample_data/soc-redditHyperlinks-title.tsv; opens a 60 MB graph
reddit_file = repo_root / "sample_data/soc-redditHyperlinks-title.tsv"
if RUN_REDDIT_DEMO:
    if not reddit_file.exists():
        raise FileNotFoundError(f"{reddit_file} not found")
    g = read_temporal_edgelist(reddit_file, delimiter="\t", header=True, columns=(0, 1, 3),
                               attrs={"sentiment": 4}, directed=True)
    print(f"{g.num_nodes:,} subreddits, {len(list(g.connectors())):,} events")
    show(g, seed=0, layout_iterations=30)

## Multiplex / multilayer graph

Each layer gets its own edge color and a checkbox in the top-right legend. Edges with no layer (the backbone) always stay visible.

Click **Layer atlas** (top-left) to see each layer as its own separated plane instead of overlaid on one — the clearest way to see "layer" as an actual dimension rather than just a color.

In [ ]:
g = Graph()
for i in range(10):
    g.add_node(i)
g.add_layer("friendship")
g.add_layer("coworker")
for i in range(10):
    g.add_edge(i, (i + 1) % 10)
for i in range(0, 10, 2):
    g.add_edge(i, (i + 3) % 10, layer="friendship")
for i in range(1, 10, 2):
    g.add_edge(i, (i + 4) % 10, layer="coworker")

show(g, seed=5)

## Hypergraph

Each hyperedge (a connector with more than 2 members) renders as a translucent convex-hull polygon wrapping its member nodes, instead of a line.

In [ ]:
g = Graph()
for i in range(12):
    g.add_node(i)
g.add_hyperedge([0, 1, 2, 3])
g.add_hyperedge([3, 4, 5])
g.add_hyperedge([6, 7, 8, 9, 10])
g.add_hyperedge([1, 6, 11])

show(g, seed=2)

## Custom styling

`show()` accepts style kwargs — no JS/CSS required. Colors are `[r, g, b, a]` in 0-1 range. See `hyperloom_bridge.show`'s docstring for the full list (node_color, node_radius_px, edge_color, background_color, arrow_color/length/width/t, hull_padding). Per-attribute styling and node shape aren't supported yet.

In [ ]:
g = Graph()
for i in range(8):
    g.add_node(i)
for i in range(7):
    g.add_edge(i, i + 1)

show(
    g,
    seed=1,
    node_color=[1.0, 0.1, 0.1, 1.0],
    node_radius_px=10,
    background_color=[0.1, 0.1, 0.15, 1.0],
    edge_color=[0.5, 0.5, 0.6, 0.6],
)

## Artistic styling, networkx-style

`show()` takes the arguments you know from `networkx.draw`, plus encodings that compute colors and sizes from the graph. The same arguments work **live** on the open viewer (next cells), and the **Appearance** panel at the top right does it all without code: colors (single, attribute, degree, weight, time, time bucket; palettes and colormaps), sizes, shapes, opacity, outlines, curved edges, labels, background, and painting the nodes you select.

A tuple is one color; a **list or array has one entry per node** (or edge); a dict maps node keys to values. Mistakes (unknown colormap, wrong count, ...) raise immediately, before anything opens.

In [ ]:
from hyperloom_bridge import (by_attribute, by_degree, by_time_bucket, by_weight, size_by_degree, size_by_weight,
                              shape_by_attribute, RESET)

rng = np.random.default_rng(3)
n = 240
g = Graph()
for i in range(n):
    g.add_node(i, group=f"g{i * 4 // n}", score=round(float(rng.gamma(2.0, 10.0)), 1))
size = n // 4
for _ in range(4 * n):
    u = int(rng.integers(n))
    v = int(rng.integers(u // size * size, (u // size + 1) * size)) if rng.random() < 0.93 else int(rng.integers(n))
    if u != v:
        g.add_edge(u, v, weight=float(rng.integers(1, 6)))

# A dark, glowing look: colormap by degree, size by degree, white outlines, faint curved edges.
show(g, seed=1, layout_iterations=80,
     node_color=by_degree("plasma"), node_size=size_by_degree((6, 26)),
     edgecolors="white", linewidths=1, edge_alpha=0.35, edge_curvature=0.15,
     background_color="#0f172a", edge_color="#94a3b8")

In [ ]:
# The matplotlib way: one number per node through a colormap, one size per node, marker letters for shapes.
scores = [node.attrs["score"] for node in g.nodes()]
show(g, seed=1, layout_iterations=80,
     node_color=scores, cmap="viridis",                    # numbers -> colormap (legend appears)
     node_size=[6 + s / 2 for s in scores],                # pixel diameters
     node_shape=shape_by_attribute("group"),               # one shape per group
     edgecolors="black", linewidths=0.8, edge_alpha=0.4)

### Change it live

`show()` in this notebook keeps the open viewer in `_active_view`. Every option above works on it, one at a time, and the viewer updates immediately. Run the next cell while the viewer is visible.

In [ ]:
import time

for curvature in (0.0, 0.15, 0.3, 0.45):
    _active_view.style(edge_curvature=curvature)          # bend the edges
    time.sleep(0.5)
_active_view.style(node_shape="s", node_alpha=0.85)        # squares, slightly transparent
time.sleep(0.5)
_active_view.color_nodes([0, 1, 2, 3, 4], "crimson")       # paint specific nodes
time.sleep(0.5)
_active_view.style(node_color=by_attribute("group", palette="Set2"))   # recolor by group (crimson nodes stay painted)
print(_active_view.get_style()["node"].keys())
# _active_view.clear_colors(); _active_view.style(node_shape=RESET); _active_view.reset_style()

### Color by time bucket

For temporal data, `by_time_bucket` splits the time span into buckets exactly like the **Time ribbon** and colors by bucket: edges by their start time, nodes by the bucket of their first (or last) activity. `split="events"` gives every bucket the same number of events instead of the same duration. The legends show how many edges each bucket holds.

In [ ]:
from hyperloom_core import from_temporal_edgelist

events = [(i % 30, (i * 7 + 3) % 30, i) for i in range(300) if i % 30 != (i * 7 + 3) % 30]
tg = from_temporal_edgelist(events)
show(tg, seed=2, layout_iterations=80,
     edge_color=by_time_bucket(6), edge_width=1.6, edge_curvature=0.12,
     node_color=by_time_bucket(6, node_time="first"), node_size=12, edgecolors="white", linewidths=1)

In [ ]:
# Edges by weight: color through a colormap and width from the same numbers.
show(g, seed=1, layout_iterations=80,
     edge_color=by_weight("Blues"), edge_width=size_by_weight((0.5, 4)), edge_alpha=0.7,
     node_color="#1e3a5f", node_size=7)

## Exploring a graph: style, filter, select, path

The panel at the top-right turns the viewer into an analysis tool. This graph has 900 people in three teams, each with a `score`.

- **Color by / Size by**: color by `team`; size by `Degree` or by `score`.
- **Filter**: pick `team` and tick one team; pick `score` and narrow its range; raise **Min degree** to keep only well-connected people. Connectors need both ends visible, and positions do not move, so the picture stays a map. **Reset filter** clears it.
- **Select & path**: click two people (or use the **+** beside a search result), then **Find path** shows the fewest-hop route between them, or says there is none. With one person selected, **Show neighbourhood** isolates them and their contacts; **Show all nodes** returns.
- **Find a node** searches by key (try `person-42`); the **+/-** buttons zoom.

In [ ]:
rng = np.random.default_rng(0)
teams = ["research", "engineering", "design"]
n = 900
g = Graph()
for i in range(n):
    g.add_node(f"person-{i}", team=teams[i * 3 // n], score=round(float(rng.gamma(2.0, 10.0)), 1))
for _ in range(3 * n):
    u = int(rng.integers(n))
    same_team = rng.random() < 0.92
    v = int(rng.integers(u // (n // 3) * (n // 3), (u // (n // 3) + 1) * (n // 3))) if same_team else int(rng.integers(n))
    if u != v:
        g.add_edge(f"person-{u}", f"person-{v}")

show(g, seed=0, node_color_by="team", layout_iterations=80)

## Large graphs: overview, drill-down, grouping

Past 5,000 visible nodes the viewer stops drawing every node and shows an **overview**: spatial cells, colored by their dominant group, with the strongest links between them (the caption at the top says how many are shown). The overview is adaptive:

- **Hover** a cell to see how many nodes it holds; **click** it, or scroll to zoom, and the cells get finer.
- Zoom far enough that fewer than 5,000 nodes are in view and you see the **individual nodes and their edges** ("Zoomed detail"). **Fit view** returns to the overview.
- **Group by** `community` (Style panel) collapses each community into one point with weighted links between them. **Click a group** to expand just that community.
- Individual nodes cannot be clicked in the overview; filter or search down to a subset first.

In [ ]:
rng = np.random.default_rng(1)
n, k = 8000, 5
g = Graph()
for i in range(n):
    g.add_node(i, community=f"c{i * k // n}")
size = n // k
for _ in range(4 * n):
    u = int(rng.integers(n))
    v = int(rng.integers(u // size * size, (u // size + 1) * size)) if rng.random() < 0.95 else int(rng.integers(n))
    if u != v:
        g.add_edge(u, v)

show(g, seed=0, node_color_by="community", layout_iterations=40)

## 100K-node scale check

Repulsion stays active through a bounded spatial approximation. Large viewers use a labeled density overview; search to inspect a node’s incident relationships. Use the evaluation notebook for measured scale checks.

In [ ]:
RUN_100K_DEMO = False  # use evaluation.ipynb for measured, bounded stress tests
if RUN_100K_DEMO:
    rng = np.random.default_rng(0)
    n = 100_000
    g = Graph()
    for i in range(n):
        g.add_node(i)
    for u, v in rng.integers(0, n, size=(200_000, 2)):
        if u != v:
            g.add_edge(int(u), int(v))
    
    show(g, seed=0, layout_iterations=60)
